## 테이블 병합 및 최종 데이터셋 생성


assessments / courses / studentInfo / studentRegistration / studentAssessment 각각의 1차 전처리 결과(`data/interim/*_processed.csv`)를 불러와 병합한다.


In [63]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()

# 노트북에서 실행할 경우 프로젝트 최상위 폴더로 이동
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

INTERIM_DIR = PROJECT_ROOT / "data" / "interim"

ASSESSMENTS_PATH = INTERIM_DIR / "assessments_processed.csv"
COURSES_PATH = INTERIM_DIR / "courses_processed.csv"
STUDENT_INFO_PATH = INTERIM_DIR / "student_info_processed.csv"
STUDENT_REGISTRATION_PATH = INTERIM_DIR / "student_registration_processed.csv"
STUDENT_ASSESSMENT_PATH = INTERIM_DIR / "student_assessment_processed.csv"

for path in [
    ASSESSMENTS_PATH,
    COURSES_PATH,
    STUDENT_INFO_PATH,
    STUDENT_REGISTRATION_PATH,
    STUDENT_ASSESSMENT_PATH,
]:
    print(path.name, "존재:", path.exists())

assessments_processed.csv 존재: True
courses_processed.csv 존재: True
student_info_processed.csv 존재: True
student_registration_processed.csv 존재: True
student_assessment_processed.csv 존재: True


In [64]:
assessments_processed = pd.read_csv(ASSESSMENTS_PATH)
courses_processed = pd.read_csv(COURSES_PATH)
student_info_processed = pd.read_csv(STUDENT_INFO_PATH)
student_registration_processed = pd.read_csv(STUDENT_REGISTRATION_PATH)
student_assessment_processed = pd.read_csv(STUDENT_ASSESSMENT_PATH)

for name, data in {
    "assessments_processed": assessments_processed,
    "courses_processed": courses_processed,
    "student_info_processed": student_info_processed,
    "student_registration_processed": student_registration_processed,
    "student_assessment_processed": student_assessment_processed,
}.items():
    print(f"{name} 크기:", data.shape)

assessments_processed 크기: (206, 6)
courses_processed 크기: (22, 3)
student_info_processed 크기: (31482, 12)
student_registration_processed 크기: (32548, 7)
student_assessment_processed 크기: (173739, 5)


### 1) courses + assessments


In [65]:
courses_assessment = pd.merge(
    courses_processed,
    assessments_processed,
    on=["code_module", "code_presentation"],
    how="inner"
)

print("courses_assessment 크기:", courses_assessment.shape)
courses_assessment.info()

courses_assessment 크기: (206, 7)
<class 'pandas.DataFrame'>
RangeIndex: 206 entries, 0 to 205
Data columns (total 7 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   code_module                 206 non-null    str    
 1   code_presentation           206 non-null    str    
 2   module_presentation_length  206 non-null    int64  
 3   id_assessment               206 non-null    int64  
 4   assessment_type             206 non-null    str    
 5   date                        206 non-null    str    
 6   weight                      206 non-null    float64
dtypes: float64(1), int64(2), str(4)
memory usage: 14.3 KB


### 2) courses + studentInfo


In [66]:
courses_student = pd.merge(
    courses_processed,
    student_info_processed,
    on=["code_module", "code_presentation"],
    how="inner"
)

print("courses_student 크기:", courses_student.shape)
courses_student.info()

courses_student 크기: (31482, 13)
<class 'pandas.DataFrame'>
RangeIndex: 31482 entries, 0 to 31481
Data columns (total 13 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   code_module                 31482 non-null  str  
 1   code_presentation           31482 non-null  str  
 2   module_presentation_length  31482 non-null  int64
 3   id_student                  31482 non-null  int64
 4   gender                      31482 non-null  str  
 5   region                      31482 non-null  str  
 6   num_of_prev_attempts        31482 non-null  int64
 7   studied_credits             31482 non-null  int64
 8   disability                  31482 non-null  str  
 9   final_result                31482 non-null  str  
 10  imd_band_cd                 31482 non-null  int64
 11  age_band_cd                 31482 non-null  str  
 12  highest_education_cd        31482 non-null  int64
dtypes: int64(6), str(7)
memory usage: 4.1 MB

### 3) (courses + studentInfo) + assessments


In [67]:
courses_student_assessments = pd.merge(
    courses_student,
    assessments_processed,
    on=["code_module", "code_presentation"],
    how="inner"
)

print("courses_student_assessments 크기:", courses_student_assessments.shape)
courses_student_assessments.info()

courses_student_assessments 크기: (312756, 17)
<class 'pandas.DataFrame'>
RangeIndex: 312756 entries, 0 to 312755
Data columns (total 17 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   code_module                 312756 non-null  str    
 1   code_presentation           312756 non-null  str    
 2   module_presentation_length  312756 non-null  int64  
 3   id_student                  312756 non-null  int64  
 4   gender                      312756 non-null  str    
 5   region                      312756 non-null  str    
 6   num_of_prev_attempts        312756 non-null  int64  
 7   studied_credits             312756 non-null  int64  
 8   disability                  312756 non-null  str    
 9   final_result                312756 non-null  str    
 10  imd_band_cd                 312756 non-null  int64  
 11  age_band_cd                 312756 non-null  str    
 12  highest_education_cd        312756 non

### 4) + studentAssessment


In [68]:
courses_student_assessments = pd.merge(
    courses_student_assessments,
    student_assessment_processed,
    on=["id_assessment", "id_student"],
    how="inner"
)

print("courses_student_assessments 크기:", courses_student_assessments.shape)
courses_student_assessments.info()

# 평가 제출 기록이 연결되지 않는 경우는 이탈 또는 미제출로 볼 수 있음
print(
    "\nscore 결측치:",
    courses_student_assessments["score"].isna().sum()
)

courses_student_assessments 크기: (166043, 20)
<class 'pandas.DataFrame'>
RangeIndex: 166043 entries, 0 to 166042
Data columns (total 20 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   code_module                 166043 non-null  str    
 1   code_presentation           166043 non-null  str    
 2   module_presentation_length  166043 non-null  int64  
 3   id_student                  166043 non-null  int64  
 4   gender                      166043 non-null  str    
 5   region                      166043 non-null  str    
 6   num_of_prev_attempts        166043 non-null  int64  
 7   studied_credits             166043 non-null  int64  
 8   disability                  166043 non-null  str    
 9   final_result                166043 non-null  str    
 10  imd_band_cd                 166043 non-null  int64  
 11  age_band_cd                 166043 non-null  str    
 12  highest_education_cd        166043 non

### 5) studentInfo + studentRegistration


In [69]:
student_registration_merged_df = pd.merge(
    student_info_processed,
    student_registration_processed,
    on=["id_student", "code_module", "code_presentation"],
    how="inner"
)

print("student_registration_merged_df 크기:", student_registration_merged_df.shape)
student_registration_merged_df.info()

student_registration_merged_df 크기: (31437, 16)
<class 'pandas.DataFrame'>
RangeIndex: 31437 entries, 0 to 31436
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   code_module           31437 non-null  str    
 1   code_presentation     31437 non-null  str    
 2   id_student            31437 non-null  int64  
 3   gender                31437 non-null  str    
 4   region                31437 non-null  str    
 5   num_of_prev_attempts  31437 non-null  int64  
 6   studied_credits       31437 non-null  int64  
 7   disability            31437 non-null  str    
 8   final_result          31437 non-null  str    
 9   imd_band_cd           31437 non-null  int64  
 10  age_band_cd           31437 non-null  str    
 11  highest_education_cd  31437 non-null  int64  
 12  date_registration     31437 non-null  int64  
 13  date_unregistration   9798 non-null   float64
 14  unregister_yn         31437 non-nu

In [70]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

STUDENT_REGISTRATION_MERGED_PATH = PROCESSED_DIR / "student_registration_merged.csv"

student_registration_merged_df.to_csv(STUDENT_REGISTRATION_MERGED_PATH, index=False)

print("저장 완료:", STUDENT_REGISTRATION_MERGED_PATH.name)

저장 완료: student_registration_merged.csv


### 6) + studentRegistration (최종 병합, left join)


In [71]:
merged = pd.merge(
    courses_student_assessments,
    student_registration_processed,
    on=["id_student", "code_module", "code_presentation"],
    how="inner"
)

print("merged 크기:", merged.shape)
merged.info()

merged 크기: (166036, 24)
<class 'pandas.DataFrame'>
RangeIndex: 166036 entries, 0 to 166035
Data columns (total 24 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   code_module                 166036 non-null  str    
 1   code_presentation           166036 non-null  str    
 2   module_presentation_length  166036 non-null  int64  
 3   id_student                  166036 non-null  int64  
 4   gender                      166036 non-null  str    
 5   region                      166036 non-null  str    
 6   num_of_prev_attempts        166036 non-null  int64  
 7   studied_credits             166036 non-null  int64  
 8   disability                  166036 non-null  str    
 9   final_result                166036 non-null  str    
 10  imd_band_cd                 166036 non-null  int64  
 11  age_band_cd                 166036 non-null  str    
 12  highest_education_cd        166036 non-null  int64  
 13  i

### 7) 병합 결과 검증


In [72]:
# 병합 결과 검증
print("최종 컬럼:")
for number, column in enumerate(merged.columns, start=1):
    print(number, column)

suffix_columns = [
    col for col in merged.columns
    if col.endswith("_x") or col.endswith("_y")
]
print("\n중복 병합 의심 컬럼:", suffix_columns)

print("\n전체 결측치:")
print(merged.isna().sum())

display(merged.head())

최종 컬럼:
1 code_module
2 code_presentation
3 module_presentation_length
4 id_student
5 gender
6 region
7 num_of_prev_attempts
8 studied_credits
9 disability
10 final_result
11 imd_band_cd
12 age_band_cd
13 highest_education_cd
14 id_assessment
15 assessment_type
16 date
17 weight
18 date_submitted
19 is_banked
20 score
21 date_registration
22 date_unregistration
23 unregister_yn
24 unregister_week

중복 병합 의심 컬럼: []

전체 결측치:
code_module                        0
code_presentation                  0
module_presentation_length         0
id_student                         0
gender                             0
region                             0
num_of_prev_attempts               0
studied_credits                    0
disability                         0
final_result                       0
imd_band_cd                        0
age_band_cd                        0
highest_education_cd               0
id_assessment                      0
assessment_type                    0
date                

,code_module,code_presentation,module_presentation_length,id_student,gender,region,num_of_prev_attempts,studied_credits,disability,final_result,...,assessment_type,date,weight,date_submitted,is_banked,score,date_registration,date_unregistration,unregister_yn,unregister_week
0,AAA,2013J,268,11391,M,East Anglian Region,0,240,N,Pass,...,TMA,19,10.0,18,0,78.0,-159,NaN,N,N
1,AAA,2013J,268,11391,M,East Anglian Region,0,240,N,Pass,...,TMA,54,20.0,53,0,85.0,-159,NaN,N,N
2,AAA,2013J,268,11391,M,East Anglian Region,0,240,N,Pass,...,TMA,117,20.0,115,0,80.0,-159,NaN,N,N
3,AAA,2013J,268,11391,M,East Anglian Region,0,240,N,Pass,...,TMA,166,20.0,164,0,85.0,-159,NaN,N,N
4,AAA,2013J,268,11391,M,East Anglian Region,0,240,N,Pass,...,TMA,215,30.0,212,0,82.0,-159,NaN,N,N


In [73]:
MERGED_OUTPUT_PATH = PROCESSED_DIR / "pre_merged.csv"

merged.to_csv(MERGED_OUTPUT_PATH, index=False)

print("저장 완료")
print(
    MERGED_OUTPUT_PATH.name,
    f"{MERGED_OUTPUT_PATH.stat().st_size / 1024**2:.2f} MB"
)

저장 완료
pre_merged.csv 15.51 MB
